In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

#### Reading data

In [0]:
df = spark.read.format("parquet")\
                .load("abfss://bronze@databrickspipeline07.dfs.core.windows.net/products")

In [0]:
df.display()

#### removing/dropping unused columns

In [0]:
df = df.drop("_rescued_data")
df.display()

#### Functions
- This is similar to sql functions that can be retained even after this session ends

In [0]:
df.createOrReplaceTempView("products")

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_pipeline_cata.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90

In [0]:
%sql
SELECT product_id, price, databricks_pipeline_cata.bronze.discount_func(price) AS discounted_price
FROM products

##### expr() -> function used to call sql function inside the code

In [0]:
df = df.withColumn("discounted_price", expr("databricks_pipeline_cata.bronze.discount_func(price)"))
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_pipeline_cata.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$

In [0]:
%sql
SELECT product_id, brand, databricks_pipeline_cata.bronze.upper_func(brand) AS brand_upper
FROM products

In [0]:
df.withColumn("upper_brand", expr("databricks_pipeline_cata.bronze.upper_func(brand)")).display()

#### Writing data

In [0]:
df.write.mode("append")\
    .format("delta")\
    .save("abfss://silver@databrickspipeline07.dfs.core.windows.net/products")

#### Creating table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_pipeline_cata.silver.products
USING DELTA
LOCATION 'abfss://silver@databrickspipeline07.dfs.core.windows.net/products'

In [0]:
%sql
SELECT * FROM databricks_pipeline_cata.silver.products